# AICE Professional 주피터라이트 실전 모의고사 06회 — 공개데이터(소득 구간 · 영문 트윗 감정 · 꽃 5종)

## [ 필독 ] AICE Professional 모의고사 유의사항

- 공식 샘플문항(가공식품 카테고리 분류)과 **같은 시험지 형식**으로 만든 모의고사입니다. 3문항, 180분 기준입니다.
- 공식 시험과 같은 **Tabular / Text / Image 3유형**입니다. 2번 Text 문항은 공식 샘플(상품명 → 카테고리)과 같은 텍스트 분류입니다.
- 본인 스스로 모델을 만들고 **달성 목표 이상의 성능**이 나오도록 개발해야 합니다. 정답(테스트 라벨)은 제공되지 않습니다.
  대신 이 노트북 **맨 마지막 셀**(전체 자동 채점, 정답 키 인코딩)로 제출 파일을 스스로 채점할 수 있습니다(연습용).

## 사용할 수 있는 라이브러리 (주피터라이트 판)

이 모의고사는 **브라우저 주피터라이트(Pyodide · Python 3.14)** 에서 설치 없이 끝까지 풀 수 있도록 라이브러리를 제한했습니다.
필요한 라이브러리는 주피터라이트에 **미리 설치**되어 있습니다. 맨 위 '브라우저 실습' 셀은 확인만 하고 바로 끝납니다(Colab·로컬 Jupyter 에서는 아무 일도 하지 않습니다).

| 구분 | 쓸 수 있음 | 쓰지 않음(브라우저에서 안 됨) |
|---|---|---|
| 데이터·그림 | numpy · pandas · matplotlib · seaborn · Pillow | — |
| 머신러닝 | scikit-learn · LightGBM · XGBoost · imbalanced-learn · joblib | CatBoost |
| 텍스트 | scikit-learn 의 CountVectorizer · TfidfVectorizer · 내장 영어 불용어, **NLTK**(불용어·토큰화·어간 추출, 데이터는 `/drive/nltk_data` 에 미리 들어 있음) | 형태소 분석기 · BERT 등 사전학습 언어 모델 |
| 딥러닝 | `tensorflow.keras` 문법(TensorFlow.js 로 실행되는 대체 구현체, 미리 설치): `Sequential`·함수형(`Input` → 층 → `Model`) · Dense · Dropout · BatchNormalization · Conv2D · MaxPooling2D · GlobalAveragePooling2D · Embedding · LSTM · GRU · 콜백 3종(EarlyStopping · ModelCheckpoint · ReduceLROnPlateau) · `load_img` · `img_to_array` · `image_dataset_from_directory` · `ImageDataGenerator` · `model.save` / `load_model` · **MobileNet v1 전이학습**(`alpha=0.25`, 224×224, `weights='imagenet'` 만) | PyTorch · 다른 사전학습 모델(MobileNetV2/V3 · ResNet · EfficientNet 등) · 사용자 정의 층·손실 · GPU |

- 이미지는 64×64 정도로 줄여 작은 CNN 을 직접 학습하거나, MobileNet v1(alpha 0.25)로 특징을 뽑아 분류기를 학습하세요(전이학습). 브라우저 TensorFlow.js 는 CPython TensorFlow 와 수치가 조금 달라 딥러닝 점수가 약간 다르게 나올 수 있습니다.
- 학습이 오래 걸리면 epoch 를 줄이거나 조기 종료(EarlyStopping)를 쓰세요. 최신 Chrome·Edge 를 권장합니다.

## 시간 배분(180분)

| 문항 | 권장 시간 | 먼저 할 일 |
|---|---|---|
| 1. Tabular (30점) | 50분 | `info()`·`describe()` 로 자료형·결측 확인 → 기본 모델로 제출 파일부터 만들기 |
| 2. Text (35점) | 60분 | 라벨 분포·문장 길이 확인 → TF-IDF + 선형 모델로 기준 점수 |
| 3. Image (35점) | 60분 | 압축 풀기·폴더 구조 확인 → 작은 CNN 으로 기준 점수 |
| 마무리 | 10분 | 제출 파일 6개 이름·열 구조 확인 → 맨 마지막 채점 셀 실행 |

## 시험 자원 가이드

| 구분 | 공식 시험 | 이 모의고사 |
|---|---|---|
| IDE | CPU 8 core · Memory 32G · GPU 1개(VRAM 16GB) | **브라우저 주피터라이트**(설치 없음, CPU) — Colab·로컬 Jupyter 에서도 그대로 실행됨 |
| SW | Python 3.10 · TensorFlow 2.13 · PyTorch 2.1 | Python 3.14(Pyodide) · scikit-learn · LightGBM · XGBoost · Keras 문법(TensorFlow.js) — 위 표의 라이브러리만 |

## 유의 사항

- 막히면 문항 아래 접혀 있는 **힌트 1·2·3단계**를 순서대로 펼쳐 보세요. 단계가 올라갈수록 정답에 가까워집니다.
- 제출 파일명은 **고정**입니다: `answer_1~3` 모델 파일과 `answer_1~3.csv`.


문제에 제시되는 가이드는 필수적으로 준수하여 주시기 바랍니다. 이를 준수하지 않을 경우 채점 제한 또는 0점 처리될 수 있으니
각별히 유의해 주시기 바랍니다.

#### **[학습 정보]**
- 학습 목표 : Tabular · Text · Image 예측 모델 3종을 만들어 달성 목표 이상의 예측 파일을 제출하고, 제출한 코드로 같은 모델이 재현되게 할 수 있다
- AICE 역량 : PT1·PT2 표 데이터 / PX1·PX2 텍스트 / PI1·PI2 이미지 / PS1 제출 파일 형식·재현성
- 제출 파일 : `answer_1~3` 모델 파일(.pkl · .h5 등)과 `answer_1~3.csv` — **파일명은 고정**입니다(이름을 바꾸면 채점되지 않음)

| 문항 | 문제 | 달성 목표 | 역량 | 기술 | 예상 시간 |
|---|---|---|---|---|---|
| 1. Tabular | 우편번호별 소득 구간 분류 | 정확도 0.9 이상 | PT1 표 데이터 전처리·피처 엔지니어링, PT2 표 데이터 모델 학습·평가 | pandas, scikit-learn(LabelEncoder), joblib | 50분 |
| 2. Text | 영문 트윗 감정 분류(4종) | 매크로 F1 0.58 이상 | PX1 텍스트 정제·벡터화, PX2 텍스트 분류 모델 학습·평가 | pandas, scikit-learn(ENGLISH_STOP_WORDS, Pipeline, CountVectorizer, TfidfVectorizer, LogisticRegression, GridSearchCV), joblib | 60분 |
| 3. Image | 꽃 종류 이미지 분류(5종) | 정확도 0.85 이상 | PI1 이미지 로딩·전처리, PI2 이미지 분류 모델(CNN·전이학습) | pandas, Keras(TensorFlow.js) | 60분 |

In [1]:
# (브라우저 실습) 주피터라이트에는 tensorflow(TensorFlow.js 대체 구현체)·scikit-learn·LightGBM 등이 미리 설치되어 있어 이 셀은 확인만 합니다.
# Colab·로컬 Jupyter 에서는 아무 일도 하지 않습니다.
import os, sys, importlib.util
if 'pyodide' in sys.modules:
    os.environ['AICE_IMG_DIR'] = '/tmp/aice_images/' + os.path.basename(os.getcwd())   # 이미지 압축은 메모리 폴더에 풂(빠름)
    if not ('tensorflow' in sys.modules or importlib.util.find_spec('tensorflow')):
        # 미리 설치되지 않은 사이트(패치 전 주피터라이트)에서만: 같은 폴더의 tfjs_keras.py 를 불러옵니다
        sys.path[:0] = [os.getcwd(), '/drive']
        import tfjs_keras
        await tfjs_keras.load()
print('준비 완료 — 다음 셀로 넘어가세요.')

준비 완료 — 다음 셀로 넘어가세요.


## 1. Tabular 문제 - 우편번호별 소득 구간 분류 (30점)

<sub>학습 목표: 우편번호별 소득 구간 분류 모델로 테스트 정확도 0.9 이상을 달성할 수 있다 | 역량: PT1 표 데이터 전처리·피처 엔지니어링, PT2 표 데이터 모델 학습·평가 | 기술: pandas, scikit-learn(LabelEncoder), joblib | 난이도: 중급 | 예상 시간: 50분</sub>

### 문제 개요

미국 국세청(IRS)의 우편번호 × 소득 구간별 신고 통계 한 행이 어느 조정총소득(AGI) 구간에 해당하는지 분류하는 문제입니다.

제공된 훈련 데이터셋을 활용해 다중 분류(6종) 모델을 개발하고, 테스트 데이터셋의 'Size of adjusted gross income' 값을 예측하시기 바랍니다.

### 달성 목표

"01_test_x.csv" 파일의 테스트 데이터셋을 기준으로 **정확도(Accuracy) 90% 이상**을 달성하는 모델을 개발하시기 바랍니다.

단, 테스트 데이터의 정답(Label 값)은 별도로 제공되지 않습니다.

### 데이터 설명

**데이터 위치 및 폴더 구조** — 문제 파일과 동일한 곳에 위치

- `01_train.csv`: 훈련 데이터셋입니다. 훈련 데이터셋으로 모델을 만드세요.
- `01_test_x.csv`: 테스트 데이터셋입니다. 라벨('Size of adjusted gross income') 값이 제외되어 있습니다.

**Label 정보**

- 'Size of adjusted gross income' : 조정총소득 구간 6종($1 under $25,000 ~ $200,000 or more)

**Data 정보**

- 'STATEFIPS / STATE / zipcode' : 주 코드·주 약자·우편번호
- 'No of returns ~ (약 145개 열)' : 신고 건수와 항목별 금액(천 달러) — 대부분 '건수·금액' 쌍

**데이터 출처** — 미국 IRS SOI 2019 우편번호별 개인소득세 통계(MIT, HF aswinaus/Individual_income_tax_dataset_2019) 무작위 5,000행

### 데이터 예시

`01_train.csv`

| STATEFIPS | STATE | zipcode | No of returns | No of single returns | No of joint returns | No of head of household returns | Number of electronically filed returns | … |
|---|---|---|---|---|---|---|---|---|
| 8 | CO | 80126 | 2210 | 1180 | 640 | 290 | 2010 | … |
| 6 | CA | 91803 | 5350 | 3850 | 940 | 470 | 4590 | … |
| 5 | AR | 71833 | 260 | 150 | 60 | 50 | 250 | … |

`01_test_x.csv`

| STATEFIPS | STATE | zipcode | No of returns | No of single returns | No of joint returns | No of head of household returns | Number of electronically filed returns | … |
|---|---|---|---|---|---|---|---|---|
| 1 | AL | 35579 | 190 | 50 | 120 | 30 | 160 | … |
| 6 | CA | 92105 | 170 | 40 | 120 | 0 | 160 | … |
| 1 | AL | 35671 | 100 | 40 | 40 | 0 | 100 | … |

### 문제 해결 가이드

(1) 훈련 데이터셋을 이용하여 우편번호별 소득 구간 분류 모델을 만드세요.
- 필요 시 추가로 함수를 만들거나 cell 을 만들어도 됩니다.
- 성능이 가장 좋은 모델을 저장하세요.
- 파일명 : **answer_1.pkl** (또는 다른 확장자의 모델 파일)

(2) 라벨('Size of adjusted gross income') 값이 제외된 테스트 데이터 파일을 읽어들여 예측값에 대한 컬럼을 추가하여 csv 로 저장하세요.
- 파일명 : **answer_1.csv**
- 테스트 데이터 파일을 읽고 **마지막 컬럼 뒤에 'Size of adjusted gross income' 컬럼 생성** 후, 모델 예측하여 예측값을 채워주세요.
- 테스트 데이터의 행 순서와 행 수를 바꾸지 마세요. 예측값은 훈련 데이터의 'Size of adjusted gross income' 와 같은 형식(원래 라벨 이름·단위)이어야 합니다.

(3) 본 문제지를 답안코드 작성된 상태로 저장하세요.
- 파일명 : 시스템매핑번호.ipynb (본 문제지 상단의 저장버튼을 눌러 저장)

### 참고

- 열이 150개가 넘고 대부분 건수·금액 쌍입니다. 금액 열은 치우침이 커서 로그 변환이 도움이 될 수 있습니다.
- 평가 지표는 **정확도(Accuracy)** 입니다.

### 채점 진행 방식 안내

- 1번 문항 배점 : 30점
- 시스템매핑번호.ipynb 실행 가능 여부 확인
- 제출한 코드로 실행했을 때, 제출한 모델 파일과 동일하게 재현되는지 확인
- answer_1.csv 로 제출된 예측 데이터를 정답지와 대조하여, **정확도(Accuracy) 81% 이상** 달성
  (공식 샘플의 '목표 70% → 채점 기준 63%' 와 같은 비율)

### 답안 작성

필요한 cell 을 추가하여 작성하세요.

In [ ]:
# 답안 작성

In [2]:

import pandas as pd
from pathlib import Path

LABEL = 'Size of adjusted gross income'

train_path = Path('01_train.csv')
test_path = Path('01_test_x.csv')

if not test_path.exists():
    test_path = Path('01_test_x(1).csv')

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)

print('현재 작업 폴더:', Path.cwd())
print('테스트 파일:', test_path.name)

print('\n학습 데이터 크기:', train.shape)
print('테스트 데이터 크기:', test.shape)

print('\n컬럼 구조 일치 여부:')
print(train.drop(columns=[LABEL]).columns.tolist() == test.columns.tolist())

print('\n자료형별 컬럼 수:')
print(train.dtypes.value_counts())

print('\n학습 데이터 결측치:', train.isna().sum().sum())
print('테스트 데이터 결측치:', test.isna().sum().sum())

print('\n소득 구간별 데이터 수:')
print(train[LABEL].value_counts())

print('\n소득 구간별 비율(%):')
print((train[LABEL].value_counts(normalize=True) * 100).round(2))

display(train.iloc[:5, :8])


현재 작업 폴더: /content
테스트 파일: 01_test_x.csv

학습 데이터 크기: (4000, 152)
테스트 데이터 크기: (1000, 151)

컬럼 구조 일치 여부:
True

자료형별 컬럼 수:
int64      149
object       2
float64      1
Name: count, dtype: int64

학습 데이터 결측치: 0
테스트 데이터 결측치: 0

소득 구간별 데이터 수:
Size of adjusted gross income
$1 under $25,000           685
$75,000 under $100,000     678
$50,000 under $75,000      670
$200,000 or more           666
$25,000 under $50,000      654
$100,000 under $200,000    647
Name: count, dtype: int64

소득 구간별 비율(%):
Size of adjusted gross income
$1 under $25,000           17.12
$75,000 under $100,000     16.95
$50,000 under $75,000      16.75
$200,000 or more           16.65
$25,000 under $50,000      16.35
$100,000 under $200,000    16.18
Name: proportion, dtype: float64


,STATEFIPS,STATE,zipcode,No of returns,No of single returns,No of joint returns,No of head of household returns,Number of electronically filed returns
0,8,CO,80126,2210,1180,640,290,2010
1,6,CA,91803,5350,3850,940,470,4590
2,5,AR,71833,260,150,60,50,250
3,8,CO,80645,360,150,170,30,330
4,5,AR,72527,100,50,20,20,80


In [3]:

import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

def make_features(df):
    X = df.drop(
        columns=[LABEL, 'STATE', 'zipcode'],
        errors='ignore'
    ).select_dtypes(include='number')

    X = np.log1p(X.clip(lower=0))
    X.columns = [f'f{i}' for i in range(X.shape[1])]

    return X

X = make_features(train)
X_test = make_features(test)

encoder = LabelEncoder()
y = encoder.fit_transform(train[LABEL])

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print('학습 데이터:', X_train.shape)
print('검증 데이터:', X_valid.shape)
print('테스트 데이터:', X_test.shape)

print('\n학습·테스트 컬럼 일치:', X.columns.equals(X_test.columns))

print('\n전처리 후 결측치:', X.isna().sum().sum())
print('전처리 후 무한대:', np.isinf(X.to_numpy()).sum())

print('\n라벨 번호 매핑:')
for i, label in enumerate(encoder.classes_):
    print(i, ':', label)

print('\n학습 라벨 분포:', np.bincount(y_train))
print('검증 라벨 분포:', np.bincount(y_valid))


학습 데이터: (3200, 149)
검증 데이터: (800, 149)
테스트 데이터: (1000, 149)

학습·테스트 컬럼 일치: True

전처리 후 결측치: 0
전처리 후 무한대: 0

라벨 번호 매핑:
0 : $1 under $25,000
1 : $100,000 under $200,000
2 : $200,000 or more
3 : $25,000 under $50,000
4 : $50,000 under $75,000
5 : $75,000 under $100,000

학습 라벨 분포: [548 518 533 523 536 542]
검증 라벨 분포: [137 129 133 131 134 136]


In [4]:

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
import pandas as pd

models = {
    'RandomForest': RandomForestClassifier(
        n_estimators=300,
        max_features='sqrt',
        random_state=42,
        n_jobs=-1
    ),
    'LightGBM': LGBMClassifier(
        n_estimators=400,
        learning_rate=0.05,
        num_leaves=15,
        verbosity=-1,
        random_state=42,
        n_jobs=-1
    ),
    'XGBoost': XGBClassifier(
        n_estimators=400,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        tree_method='hist',
        eval_metric='mlogloss',
        random_state=42,
        n_jobs=-1
    )
}

cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

results = []

for name, model in models.items():
    cv_scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring='accuracy',
        n_jobs=1
    )

    model.fit(X_train, y_train)
    pred = model.predict(X_valid)
    valid_acc = accuracy_score(y_valid, pred)

    results.append({
        '모델': name,
        'CV 평균 정확도': cv_scores.mean(),
        'CV 표준편차': cv_scores.std(),
        '검증 정확도': valid_acc
    })

result_df = pd.DataFrame(results)
result_df = result_df.sort_values(
    '검증 정확도',
    ascending=False
)

display(result_df.round(4))

best_name = result_df.iloc[0]['모델']

print('\n현재 검증 정확도 1위:', best_name)
print('최고 검증 정확도:', round(result_df.iloc[0]['검증 정확도'], 4))


,모델,CV 평균 정확도,CV 표준편차,검증 정확도
2,XGBoost,0.9562,0.0046,0.9738
1,LightGBM,0.9594,0.0062,0.9700
0,RandomForest,0.9650,0.0019,0.9688



현재 검증 정확도 1위: XGBoost
최고 검증 정확도: 0.9738


In [5]:

import joblib
import numpy as np
from pathlib import Path
from xgboost import XGBClassifier

model_path = Path('candidate_1.pkl')
csv_path = Path('candidate_1.csv')

if model_path.exists() or csv_path.exists():
    raise FileExistsError('후보 파일이 이미 존재합니다. 기존 파일을 확인하세요.')

final_model = XGBClassifier(
    n_estimators=400,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    tree_method='hist',
    eval_metric='mlogloss',
    random_state=42,
    n_jobs=-1
)

final_model.fit(X, y)

pred_before = final_model.predict(X_test)

joblib.dump({
    'model': final_model,
    'encoder': encoder,
    'columns': X.columns.tolist()
}, model_path)

saved = joblib.load(model_path)

pred_after = saved['model'].predict(
    X_test[saved['columns']]
)

same = np.array_equal(pred_before, pred_after)
print('저장 전후 예측 일치:', same)

pred_labels = saved['encoder'].inverse_transform(
    pred_after.astype(int)
)

submission = test.copy()
submission[LABEL] = pred_labels

submission.to_csv(
    csv_path,
    index=False,
    encoding='utf-8-sig'
)

check = pd.read_csv(csv_path)

print('모델 파일 생성:', model_path.exists())
print('CSV 파일 생성:', csv_path.exists())
print('CSV 크기:', check.shape)
print('컬럼 구조 일치:',
      check.columns.tolist() == test.columns.tolist() + [LABEL])
print('원본 테스트 행 순서 유지:',
      check[test.columns].equals(test))
print('예측 결측치:', check[LABEL].isna().sum())
print('예측 라벨 정상:',
      check[LABEL].isin(encoder.classes_).all())

print('\n테스트 예측 라벨 분포:')
print(check[LABEL].value_counts())

display(check.tail())


저장 전후 예측 일치: True
모델 파일 생성: True
CSV 파일 생성: True
CSV 크기: (1000, 152)
컬럼 구조 일치: True
원본 테스트 행 순서 유지: True
예측 결측치: 0
예측 라벨 정상: True

테스트 예측 라벨 분포:
Size of adjusted gross income
$200,000 or more           192
$1 under $25,000           172
$75,000 under $100,000     170
$50,000 under $75,000      161
$25,000 under $50,000      159
$100,000 under $200,000    146
Name: count, dtype: int64


,STATEFIPS,STATE,zipcode,No of returns,No of single returns,No of joint returns,No of head of household returns,Number of electronically filed returns,Number of computer prepared paper returns,Number of returns with paid preparer's signature,...,Net investment income tax amount,Number of returns with tax due at time of filing,Tax due at time of filing amount,Number of returns with total overpayments,Total overpayments amount,Number of returns with overpayments refunded,Overpayments refunded amount,Number of returns with credit to next years estimated tax,Credited to next years estimated tax amount,Size of adjusted gross income
995,4,AZ,85172,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,"$200,000 or more"
996,6,CA,95111,3640,800,2460,330,3280,260,2430,...,0,1510,6238,2130,8175,2110,7951,30,224,"$100,000 under $200,000"
997,1,AL,36750,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,"$200,000 or more"
998,8,CO,80033,1950,500,1370,70,1770,90,990,...,0,800,3588,1150,4556,1070,3929,90,627,"$100,000 under $200,000"
999,6,CA,94806,5590,2450,1790,1220,4970,340,3470,...,0,1450,3373,4130,10776,4110,10671,30,105,"$50,000 under $75,000"


In [6]:

from pathlib import Path
import shutil
import pandas as pd
import joblib

files = {
    'candidate_1.pkl': 'answer_1.pkl',
    'candidate_1.csv': 'answer_1.csv'
}

for src, dst in files.items():
    if not Path(src).exists():
        raise FileNotFoundError(src)
    if Path(dst).exists():
        raise FileExistsError(f'기존 제출 파일 보호: {dst}')

for src, dst in files.items():
    shutil.copy2(src, dst)
    print('제출 파일 생성:', dst)

saved_model = joblib.load('answer_1.pkl')
answer = pd.read_csv('answer_1.csv')

pred = saved_model['model'].predict(
    X_test[saved_model['columns']]
)
labels = saved_model['encoder'].inverse_transform(
    pred.astype(int)
)

print('모델 재불러오기:', '성공')
print('CSV 크기:', answer.shape)
print('컬럼 구조 일치:',
      answer.columns.tolist() == test.columns.tolist() + [LABEL])
print('행 순서 일치:',
      answer[test.columns].equals(test))
print('모델과 CSV 예측 일치:',
      np.array_equal(answer[LABEL].to_numpy(), labels))


제출 파일 생성: answer_1.pkl
제출 파일 생성: answer_1.csv
모델 재불러오기: 성공
CSV 크기: (1000, 152)
컬럼 구조 일치: True
행 순서 일치: True
모델과 CSV 예측 일치: True


<details><summary>힌트 1단계 (접근 방향) - 1번 문항</summary>

클래스 분포를 확인하고, 범주형은 원-핫 인코딩합니다. 값의 크기가 다르면 선형 모델 앞에 표준화를 두고, 트리 모델과 **교차검증 정확도** 로 비교하세요.

- 이 문항의 핵심: 열이 150개가 넘고 대부분 건수·금액 쌍입니다. 금액 열은 치우침이 커서 로그 변환이 도움이 될 수 있습니다.

</details>

<details><summary>힌트 2단계 (사용할 도구) - 1번 문항</summary>

사용할 도구: `pandas`, `joblib`, `lightgbm`, `train_test_split`, `LabelEncoder`, `accuracy_score`.

- 참고 문서: <https://joblib.readthedocs.io/en/stable/generated/joblib.dump.html>
- 참고 문서: <https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html>
- 참고 문서: <https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.LabelEncoder.html>
- 참고 문서: <https://scikit-learn.org/stable/modules/generated/sklearn.metrics.accuracy_score.html>

</details>

<details><summary>힌트 3단계 (코드 구조) - 1번 문항</summary>

코드의 **앞부분은 구조만** 보여 줍니다(`____` 자리는 가이드와 공식 문서를 보고 채우세요). **뒷부분은 주석으로 할 일만** 적었습니다 — 주석을 따라 직접 작성하세요.
```python
import ____
import ____
import ____
from sklearn.model_selection import ____
from sklearn.preprocessing import ____
from sklearn.metrics import ____

LABEL = '____'
train = pd.read_csv('01_train.csv')
test = pd.read_csv('01_test_x.csv')

def make_features(df):
    X = df.drop(columns=[LABEL, '____', '____'], errors='____').select_dtypes('____')
    X = np.log1p(X.clip(lower=____))
    X.columns = [f'____' for i in range(X.shape[____])]     # 특수문자가 든 열 이름은 LightGBM 이 받지 않음
    return X

# ▷ 학습(검증 데이터·조기 종료)
# ▷ 모델 파일 저장 — answer_N.pkl / .h5
# ▷ 테스트 데이터 예측(행 순서 그대로)
# ▷ 예측 CSV 저장 — answer_N.csv(테스트 열 + 마지막 정답 열)
```
- 제출 파일명: `answer_1.pkl`(모델) · `answer_1.csv`(예측)

</details>

## 2. Text 문제 - 영문 트윗 감정 분류(4종) (35점)

<sub>학습 목표: 영문 트윗 감정 분류(4종) 모델로 테스트 매크로 F1 0.58 이상을 달성할 수 있다 | 역량: PX1 텍스트 정제·벡터화, PX2 텍스트 분류 모델 학습·평가 | 기술: pandas, scikit-learn(ENGLISH_STOP_WORDS, Pipeline, CountVectorizer, TfidfVectorizer, LogisticRegression, GridSearchCV), joblib | 난이도: 실전 | 예상 시간: 60분</sub>

### 문제 개요

영어 트윗 문장으로 글쓴이의 **감정 4종**(분노·기쁨·낙관·슬픔)을 분류하는 감정 분석 문제입니다. 공식 출제 사례(트윗 감정 4종)와 같은 형식이며, 여론·민원 게시글 모니터링에 쓰이는 기법입니다.

제공된 훈련 데이터셋을 활용해 텍스트 다중 분류(감정 4종) 모델을 개발하고, 테스트 데이터셋의 'emotion' 값을 예측하시기 바랍니다.

### 달성 목표

"02_test_x.csv" 파일의 테스트 데이터셋을 기준으로 **매크로 F1 0.58 이상**을 달성하는 모델을 개발하시기 바랍니다.

단, 테스트 데이터의 정답(Label 값)은 별도로 제공되지 않습니다.

### 데이터 설명

**데이터 위치 및 폴더 구조** — 문제 파일과 동일한 곳에 위치

- `02_train.csv`: 훈련 데이터셋입니다. 훈련 데이터셋으로 모델을 만드세요.
- `02_test_x.csv`: 테스트 데이터셋입니다. 라벨('emotion') 값이 제외되어 있습니다.

**Label 정보**

- 'emotion' : anger(분노) / joy(기쁨) / optimism(낙관) / sadness(슬픔)

**Data 정보**

- 'id' : 테스트 행 번호(테스트에만)
- 'text' : 트윗 본문(사용자 이름은 @user 로 가려져 있음, 해시태그·이모지 포함)

**데이터 출처** — TweetEval emotion(SemEval-2018 Task 1, HF cardiffnlp/tweet_eval — 연구용, 트위터 약관 적용) — 원 데이터의 train+validation / test 분리 그대로

### 데이터 예시

`02_train.csv`

| text | emotion |
|---|---|
| The fact that we have a presidential candidate that speaks the way Trump does is alarming. I thought higher of my peers. #FDT16 | anger |
| @user :)) im now writing abt the changing face of sex industry in tr :))) now men will talk to me then! #revenge | anger |
| LOL! @user was just awarded the “F” bomb trophy on #AutoDealerLive, @user :) #serious | joy |

`02_test_x.csv`

| id | text |
|---|---|
| 1 | The excitement I get when I see tipping point is on tv is rather sad |
| 2 | Not all people are #concern, some are just enjoying seeing people in #miserable. |
| 3 | It's an #outrage that airports don't have smoking rooms in the departures area. |

### 문제 해결 가이드

(1) 훈련 데이터셋을 이용하여 영문 트윗 감정 분류(4종) 모델을 만드세요.
- 필요 시 추가로 함수를 만들거나 cell 을 만들어도 됩니다.
- 성능이 가장 좋은 모델을 저장하세요.
- 파일명 : **answer_2.pkl** (또는 다른 확장자의 모델 파일)

(2) 라벨('emotion') 값이 제외된 테스트 데이터 파일을 읽어들여 예측값에 대한 컬럼을 추가하여 csv 로 저장하세요.
- 파일명 : **answer_2.csv**
- 테스트 데이터 파일을 읽고 **마지막 컬럼 뒤에 'emotion' 컬럼 생성** 후, 모델 예측하여 예측값을 채워주세요.
- 테스트 데이터의 행 순서와 행 수를 바꾸지 마세요. 예측값은 훈련 데이터의 'emotion' 와 같은 형식(원래 라벨 이름·단위)이어야 합니다.

(3) 본 문제지를 답안코드 작성된 상태로 저장하세요.
- 파일명 : 시스템매핑번호.ipynb (본 문제지 상단의 저장버튼을 눌러 저장)

### 참고

- 영문은 **소문자 변환**, URL·@user·숫자 정리, 불용어 처리(scikit-learn 내장 `ENGLISH_STOP_WORDS` — 브라우저판이라 NLTK 는 쓰지 않음)를 먼저 해 보세요. 해시태그(#sad 등)는 감정 단서라 지우지 않는 편이 낫습니다.
- `CountVectorizer`(단어 개수)와 `TfidfVectorizer` 를 비교하고, 단어 1~2gram 을 써 보세요.
- 학습 라벨 분포: anger 1560, sadness 944, joy 805, optimism 322 — 불균형이라 평가 지표는 **매크로 F1**(4종 F1 의 평균)입니다.
- 테스트 행 순서(`id`)를 그대로 유지하고, 예측값은 원래 라벨 문자열(anger 등)로 제출하세요.

### 채점 진행 방식 안내

- 2번 문항 배점 : 35점
- 시스템매핑번호.ipynb 실행 가능 여부 확인
- 제출한 코드로 실행했을 때, 제출한 모델 파일과 동일하게 재현되는지 확인
- answer_2.csv 로 제출된 예측 데이터를 정답지와 대조하여, **매크로 F1 0.522 이상** 달성
  (공식 샘플의 '목표 70% → 채점 기준 63%' 와 같은 비율)

### 답안 작성

필요한 cell 을 추가하여 작성하세요.

In [ ]:
# 답안 작성

In [8]:

import pandas as pd

train_2 = pd.read_csv('02_train.csv')
test_2 = pd.read_csv('02_test_x.csv')

print('학습 데이터:', train_2.shape)
print('테스트 데이터:', test_2.shape)

print('\n학습 컬럼:', train_2.columns.tolist())
print('테스트 컬럼:', test_2.columns.tolist())

print('\n결측치:')
print(train_2.isna().sum())
print(test_2.isna().sum())

print('\n감정별 데이터 수:')
print(train_2['emotion'].value_counts())

print('\n감정별 비율(%):')
print((train_2['emotion'].value_counts(normalize=True) * 100).round(2))

train_2['text_length'] = train_2['text'].str.len()

print('\n트윗 글자 수 통계:')
print(train_2['text_length'].describe().round(2))

print('\n테스트 ID 중복:', test_2['id'].duplicated().sum())
print('테스트 ID 순서:', test_2['id'].head(10).tolist())

display(train_2[['text', 'emotion', 'text_length']].head())


학습 데이터: (3631, 2)
테스트 데이터: (1421, 2)

학습 컬럼: ['text', 'emotion']
테스트 컬럼: ['id', 'text']

결측치:
text       0
emotion    0
dtype: int64
id      0
text    0
dtype: int64

감정별 데이터 수:
emotion
anger       1560
sadness      944
joy          805
optimism     322
Name: count, dtype: int64

감정별 비율(%):
emotion
anger       42.96
sadness     26.00
joy         22.17
optimism     8.87
Name: proportion, dtype: float64

트윗 글자 수 통계:
count    3631.00
mean       91.00
std        36.73
min         6.00
25%        60.00
50%        95.00
75%       125.00
max       160.00
Name: text_length, dtype: float64

테스트 ID 중복: 0
테스트 ID 순서: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]


,text,emotion,text_length
0,The fact that we have a presidential candidate...,anger,127
1,@user :)) im now writing abt the changing face...,anger,112
2,LOL! @user was just awarded the “F” bomb troph...,joy,85
3,@user @user horrible experience with a compan...,anger,123
4,@user jeezus God #dark,sadness,22


In [9]:

import re
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

STOP = set(ENGLISH_STOP_WORDS) - {
    'not', 'no', 'nor', 'never', 'none'
}

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    text = re.sub(r'@user\b', ' ', text)
    text = re.sub(r'\d+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

X_text = train_2['text'].apply(clean_text)
X_test_text = test_2['text'].apply(clean_text)
y_text = train_2['emotion']

X_train_2, X_valid_2, y_train_2, y_valid_2 = train_test_split(
    X_text,
    y_text,
    test_size=0.2,
    random_state=42,
    stratify=y_text
)

print('학습 데이터:', len(X_train_2))
print('검증 데이터:', len(X_valid_2))
print('테스트 데이터:', len(X_test_text))

print('\n학습 감정 분포:')
print(y_train_2.value_counts())

print('\n검증 감정 분포:')
print(y_valid_2.value_counts())

print('\n전처리 후 빈 문장:', (X_text.str.len() == 0).sum())
print('부정어 보존 확인:', all(w not in STOP for w in [
    'not', 'no', 'nor', 'never', 'none'
]))

print('\n전처리 예시:')
for i in range(3):
    print('원본:', train_2['text'].iloc[i])
    print('변환:', X_text.iloc[i])
    print()


학습 데이터: 2904
검증 데이터: 727
테스트 데이터: 1421

학습 감정 분포:
emotion
anger       1248
sadness      755
joy          644
optimism     257
Name: count, dtype: int64

검증 감정 분포:
emotion
anger       312
sadness     189
joy         161
optimism     65
Name: count, dtype: int64

전처리 후 빈 문장: 0
부정어 보존 확인: True

전처리 예시:
원본: The fact that we have a presidential candidate that speaks the way Trump does is alarming. I thought higher of my peers. #FDT16
변환: the fact that we have a presidential candidate that speaks the way trump does is alarming. i thought higher of my peers. #fdt

원본: @user :)) im now writing abt the changing face of sex industry in tr :))) now men will talk to me then! #revenge
변환: :)) im now writing abt the changing face of sex industry in tr :))) now men will talk to me then! #revenge

원본: LOL! @user was just awarded the “F” bomb trophy on #AutoDealerLive, @user :) #serious
변환: lol! was just awarded the “f” bomb trophy on #autodealerlive, :) #serious



In [10]:

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import f1_score, classification_report
import pandas as pd

models_2 = {
    'Word + LogisticRegression': Pipeline([
        ('tfidf', TfidfVectorizer(
            ngram_range=(1, 2),
            min_df=2,
            stop_words=list(STOP),
            sublinear_tf=True
        )),
        ('model', LogisticRegression(
            max_iter=2000,
            class_weight='balanced',
            random_state=42
        ))
    ]),

    'Word + LinearSVC': Pipeline([
        ('tfidf', TfidfVectorizer(
            ngram_range=(1, 2),
            min_df=2,
            stop_words=list(STOP),
            sublinear_tf=True
        )),
        ('model', LinearSVC(
            C=1.0,
            class_weight='balanced',
            random_state=42
        ))
    ]),

    'Char + LinearSVC': Pipeline([
        ('tfidf', TfidfVectorizer(
            analyzer='char_wb',
            ngram_range=(2, 5),
            min_df=2,
            sublinear_tf=True
        )),
        ('model', LinearSVC(
            C=1.0,
            class_weight='balanced',
            random_state=42
        ))
    ]),

    'Char + LogisticRegression': Pipeline([
        ('tfidf', TfidfVectorizer(
            analyzer='char_wb',
            ngram_range=(2, 5),
            min_df=2,
            sublinear_tf=True
        )),
        ('model', LogisticRegression(
            max_iter=2000,
            class_weight='balanced',
            random_state=42
        ))
    ])
}

cv_2 = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

results_2 = []

for name, model in models_2.items():
    cv_scores = cross_val_score(
        model,
        X_train_2,
        y_train_2,
        cv=cv_2,
        scoring='f1_macro',
        n_jobs=1
    )

    model.fit(X_train_2, y_train_2)
    pred = model.predict(X_valid_2)

    results_2.append({
        '모델': name,
        'CV Macro F1': cv_scores.mean(),
        'CV 표준편차': cv_scores.std(),
        '검증 Macro F1': f1_score(
            y_valid_2, pred, average='macro'
        )
    })

result_df_2 = pd.DataFrame(results_2).sort_values(
    '검증 Macro F1',
    ascending=False
)

display(result_df_2.round(4))

best_name_2 = result_df_2.iloc[0]['모델']
best_model_2 = models_2[best_name_2]
best_pred_2 = best_model_2.predict(X_valid_2)

print('\n검증 1위 모델:', best_name_2)

print('\n감정별 성능:')
print(classification_report(
    y_valid_2,
    best_pred_2,
    digits=4,
    zero_division=0
))


,모델,CV Macro F1,CV 표준편차,검증 Macro F1
3,Char + LogisticRegression,0.6416,0.0095,0.6457
0,Word + LogisticRegression,0.6397,0.0065,0.6443
1,Word + LinearSVC,0.6308,0.0125,0.6310
2,Char + LinearSVC,0.6367,0.0022,0.6141



검증 1위 모델: Char + LogisticRegression

감정별 성능:
              precision    recall  f1-score   support

       anger     0.7942    0.7917    0.7929       312
         joy     0.6753    0.6460    0.6603       161
    optimism     0.5000    0.4615    0.4800        65
     sadness     0.6287    0.6720    0.6496       189

    accuracy                         0.6988       727
   macro avg     0.6496    0.6428    0.6457       727
weighted avg     0.6986    0.6988    0.6983       727



In [11]:

import joblib
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.base import clone

model_path_2 = Path('candidate_2.pkl')
csv_path_2 = Path('candidate_2.csv')

if model_path_2.exists() or csv_path_2.exists():
    raise FileExistsError('후보 파일이 이미 존재합니다.')

final_model_2 = clone(models_2['Char + LogisticRegression'])

final_model_2.fit(X_text, y_text)

pred_before_2 = final_model_2.predict(X_test_text)

joblib.dump(final_model_2, model_path_2)

loaded_model_2 = joblib.load(model_path_2)
pred_after_2 = loaded_model_2.predict(X_test_text)

print('저장 전후 예측 일치:',
      np.array_equal(pred_before_2, pred_after_2))

submission_2 = test_2.copy()
submission_2['emotion'] = pred_after_2

submission_2.to_csv(
    csv_path_2,
    index=False,
    encoding='utf-8-sig'
)

check_2 = pd.read_csv(csv_path_2)

print('모델 파일 생성:', model_path_2.exists())
print('CSV 파일 생성:', csv_path_2.exists())
print('CSV 크기:', check_2.shape)

print('컬럼 구조 일치:',
      check_2.columns.tolist() == ['id', 'text', 'emotion'])

print('테스트 행 순서 유지:',
      check_2[['id', 'text']].equals(test_2))

print('예측 결측치:', check_2['emotion'].isna().sum())

print('예측 라벨 정상:',
      check_2['emotion'].isin(y_text.unique()).all())

print('\n테스트 예측 감정 분포:')
print(check_2['emotion'].value_counts())

display(check_2.head())


저장 전후 예측 일치: True
모델 파일 생성: True
CSV 파일 생성: True
CSV 크기: (1421, 3)
컬럼 구조 일치: True
테스트 행 순서 유지: True
예측 결측치: 0
예측 라벨 정상: True

테스트 예측 감정 분포:
emotion
anger       564
sadness     360
joy         340
optimism    157
Name: count, dtype: int64


,id,text,emotion
0,1,The excitement I get when I see tipping point ...,sadness
1,2,"Not all people are #concern, some are just enj...",anger
2,3,It's an #outrage that airports don't have smok...,anger
3,4,Odd watching #Antifa extremists going full spe...,anger
4,5,@user It's kay. They were my main healer/caste...,sadness


In [12]:

from pathlib import Path
import shutil
import joblib
import pandas as pd
import numpy as np

files_2 = {
    'candidate_2.pkl': 'answer_2.pkl',
    'candidate_2.csv': 'answer_2.csv'
}

for src, dst in files_2.items():
    if not Path(src).exists():
        raise FileNotFoundError(src)
    if Path(dst).exists():
        raise FileExistsError(f'기존 파일 보호: {dst}')

for src, dst in files_2.items():
    shutil.copy2(src, dst)
    print('제출 파일 생성:', dst)

saved_model_2 = joblib.load('answer_2.pkl')
answer_2 = pd.read_csv('answer_2.csv')

pred_2 = saved_model_2.predict(X_test_text)

print('모델 재불러오기: 성공')
print('CSV 크기:', answer_2.shape)
print('컬럼 구조 일치:',
      answer_2.columns.tolist() == ['id', 'text', 'emotion'])
print('테스트 행 순서 유지:',
      answer_2[['id', 'text']].equals(test_2))
print('모델과 CSV 예측 일치:',
      np.array_equal(answer_2['emotion'].to_numpy(), pred_2))
print('예측 결측치:', answer_2['emotion'].isna().sum())
print('예측 라벨 정상:',
      answer_2['emotion'].isin(y_text.unique()).all())

print('\n제출 파일 확인:')
for filename in ['answer_1.pkl', 'answer_1.csv',
                 'answer_2.pkl', 'answer_2.csv']:
    print(filename, Path(filename).exists())


제출 파일 생성: answer_2.pkl
제출 파일 생성: answer_2.csv
모델 재불러오기: 성공
CSV 크기: (1421, 3)
컬럼 구조 일치: True
테스트 행 순서 유지: True
모델과 CSV 예측 일치: True
예측 결측치: 0
예측 라벨 정상: True

제출 파일 확인:
answer_1.pkl True
answer_1.csv True
answer_2.pkl True
answer_2.csv True


<details><summary>힌트 1단계 (접근 방향) - 2번 문항</summary>

라벨 분포와 문장 길이를 확인합니다. 문장을 TF-IDF 로 바꾸되 **단어 단위와 글자 조각(char n-gram)** 을 비교하고, 나이브 베이즈·선형 SVM·로지스틱 회귀 중 **교차검증 정확도** 가 높은 조합으로 전체 학습하세요. 다른 컬럼(숫자·범주)이 단서이면 `ColumnTransformer` 로 함께 넣습니다.

- 이 문항의 핵심: 영문은 **소문자 변환**, URL·@user·숫자 정리, 불용어 처리(scikit-learn 내장 `ENGLISH_STOP_WORDS` — 브라우저판이라 NLTK 는 쓰지 않음)를 먼저 해 보세요. 해시태그(#sad 등)는 감정 단서라 지우지 않는 편이 낫습니다.

</details>

<details><summary>힌트 2단계 (사용할 도구) - 2번 문항</summary>

사용할 도구: `pandas`, `joblib`, `ENGLISH_STOP_WORDS`, `Pipeline`, `CountVectorizer`, `TfidfVectorizer`, `LogisticRegression`, `GridSearchCV`, `StratifiedKFold`, `f1_score`.

- 참고 문서: <https://joblib.readthedocs.io/en/stable/generated/joblib.dump.html>
- 참고 문서: <https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.ENGLISH_STOP_WORDS.html>
- 참고 문서: <https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html>
- 참고 문서: <https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html>
- 참고 문서: <https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html>
- 참고 문서: <https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html>

</details>

<details><summary>힌트 3단계 (코드 구조) - 2번 문항</summary>

코드의 **앞부분은 구조만** 보여 줍니다(`____` 자리는 가이드와 공식 문서를 보고 채우세요). **뒷부분은 주석으로 할 일만** 적었습니다 — 주석을 따라 직접 작성하세요.
```python
import ____
import ____
import ____
import ____
from sklearn.feature_extraction.text import ____
from sklearn.pipeline import ____
from sklearn.feature_extraction.text import ____, ____
from sklearn.linear_model import ____
from sklearn.model_selection import ____, ____
from sklearn.metrics import ____

STOP = set(ENGLISH_STOP_WORDS) - {'____', '____', '____', '____', '____'}   # scikit-learn 내장 불용어(부정어는 감정 단서라 남김)

train = pd.read_csv('02_train.csv')

# ▷ 후보 모델을 교차검증으로 비교(점수 출력)
# ▷ 학습(검증 데이터·조기 종료)
# ▷ 모델 파일 저장 — answer_N.pkl / .h5
# ▷ 테스트 데이터 예측(행 순서 그대로)
# ▷ 예측 CSV 저장 — answer_N.csv(테스트 열 + 마지막 정답 열)
```
- 제출 파일명: `answer_2.pkl`(모델) · `answer_2.csv`(예측)

</details>

## 3. Image 문제 - 꽃 종류 이미지 분류(5종) (35점)

<sub>학습 목표: 꽃 종류 이미지 분류(5종) 모델로 테스트 정확도 0.85 이상을 달성할 수 있다 | 역량: PI1 이미지 로딩·전처리, PI2 이미지 분류 모델(CNN·전이학습) | 기술: pandas, Keras(TensorFlow.js) | 난이도: 실전 | 예상 시간: 60분</sub>

### 문제 개요

꽃 사진(128×128 컬러)으로 **꽃 종류 5종** 을 분류하는 문제입니다. 공식 출제 사례(꽃 분류)와 같은 형식입니다. 학습 이미지가 종류마다 약 150~200장으로 적어 증강(뒤집기)과 Dropout·조기 종료로 과적합을 막는 것이 중요합니다.

제공된 훈련 데이터셋을 활용해 이미지 다중 분류(5종) 모델을 개발하고, 테스트 데이터셋의 'label' 값을 예측하시기 바랍니다.

### 달성 목표

"03_test_x.csv" 파일의 테스트 데이터셋을 기준으로 **정확도(Accuracy) 85% 이상**을 달성하는 모델을 개발하시기 바랍니다.

단, 테스트 데이터의 정답(Label 값)은 별도로 제공되지 않습니다.

### 데이터 설명

**데이터 위치 및 폴더 구조** — 문제 파일과 동일한 곳에 위치

- `03_images.zip`: 압축을 풀면 `03_train/flower_*.png`(학습, 폴더 구분 없음)와 `03_test/test_*.png`(테스트)가 생깁니다. 128×128 컬러 PNG.
- `03_train.csv`: 학습 이미지 파일명과 라벨
- `03_test_x.csv`: 테스트 이미지 파일명(라벨 제외)

**Label 정보**

- 'label' : passion_flower: 시계꽃 / petunia: 페튜니아 / wallflower: 꽃무(월플라워) / water_lily: 수련 / watercress: 물냉이

**데이터 출처** — Oxford Flowers-102(VGG, HF dpdl-benchmark/oxford_flowers102 — 연구용) 중 장수가 많은 5종, 가운데 정사각형 128×128

### 데이터 예시

학습 이미지 라벨별 장수: petunia 206, passion_flower 201, wallflower 157, water_lily 155, watercress 147

| file_name | label |
|---|---|
| flower_00001.png | watercress |
| flower_00002.png | wallflower |
| flower_00003.png | petunia |

### 문제 해결 가이드

(1) 훈련 데이터셋을 이용하여 꽃 종류 이미지 분류(5종) 모델을 만드세요.
- 필요 시 추가로 함수를 만들거나 cell 을 만들어도 됩니다.
- 성능이 가장 좋은 모델을 저장하세요.
- 파일명 : **answer_3.h5** (또는 다른 확장자의 모델 파일)

(2) 라벨('label') 값이 제외된 테스트 데이터 파일을 읽어들여 예측값에 대한 컬럼을 추가하여 csv 로 저장하세요.
- 파일명 : **answer_3.csv**
- 테스트 데이터 파일을 읽고 **마지막 컬럼 뒤에 'label' 컬럼 생성** 후, 모델 예측하여 예측값을 채워주세요.
- 테스트 데이터의 행 순서와 행 수를 바꾸지 마세요. 예측값은 훈련 데이터의 'label' 와 같은 형식(원래 라벨 이름·단위)이어야 합니다.

(3) 본 문제지를 답안코드 작성된 상태로 저장하세요.
- 파일명 : 시스템매핑번호.ipynb (본 문제지 상단의 저장버튼을 눌러 저장)

### 참고

- 학습 이미지가 클래스별 폴더가 아니라 한 폴더에 있고 라벨은 CSV 에 있습니다. CSV 의 `file_name` 으로 `load_img` 해서 배열로 만드세요.
- 이 모의고사는 **브라우저(주피터라이트)판** 이라 사전학습 모델(MobileNet 등)을 쓰지 않습니다. 작은 CNN(`Sequential` + `Conv2D`)을 직접 학습합니다.
- 128×128 을 그대로 쓰면 브라우저가 느려집니다. `target_size=(64, 64)` 로 줄여 읽고 1/255 로 나눠 0~1 로 맞추세요.
- 평가 지표는 **정확도(Accuracy)** 입니다. 테스트는 `03_test_x.csv` 의 파일명 순서대로 읽어 예측하세요.

### 채점 진행 방식 안내

- 3번 문항 배점 : 35점
- 시스템매핑번호.ipynb 실행 가능 여부 확인
- 제출한 코드로 실행했을 때, 제출한 모델 파일과 동일하게 재현되는지 확인
- answer_3.csv 로 제출된 예측 데이터를 정답지와 대조하여, **정확도(Accuracy) 76.5% 이상** 달성
  (공식 샘플의 '목표 70% → 채점 기준 63%' 와 같은 비율)

### 답안 작성

필요한 cell 을 추가하여 작성하세요.

In [ ]:
# 답안 작성

In [14]:

import os
import zipfile
import pandas as pd
from pathlib import Path
from PIL import Image

BASE_DIR = Path.cwd()
ZIP_PATH = BASE_DIR / '03_images.zip'

train_csv_3 = pd.read_csv('03_train.csv')
test_csv_3 = pd.read_csv('03_test_x.csv')

train_dir = BASE_DIR / '03_train'
test_dir = BASE_DIR / '03_test'

if not train_dir.exists() or not test_dir.exists():
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall(BASE_DIR)

train_files = list(train_dir.glob('*.png'))
test_files = list(test_dir.glob('*.png'))

print('현재 작업 폴더:', BASE_DIR)
print('학습 CSV 크기:', train_csv_3.shape)
print('테스트 CSV 크기:', test_csv_3.shape)

print('\n실제 학습 이미지:', len(train_files))
print('실제 테스트 이미지:', len(test_files))

missing_train = [
    f for f in train_csv_3['file_name']
    if not (train_dir / f).is_file()
]

missing_test = [
    f for f in test_csv_3['file_name']
    if not (test_dir / f).is_file()
]

print('\n누락된 학습 이미지:', len(missing_train))
print('누락된 테스트 이미지:', len(missing_test))

with Image.open(train_dir / train_csv_3['file_name'].iloc[0]) as img:
    print('\n이미지 크기:', img.size)
    print('이미지 모드:', img.mode)

print('\n꽃 종류별 데이터 수:')
print(train_csv_3['label'].value_counts())

print('\n결측치:')
print(train_csv_3.isna().sum())
print(test_csv_3.isna().sum())

print('\n학습 CSV 예시:')
display(train_csv_3.head())

print('\n테스트 CSV 예시:')
display(test_csv_3.head())


현재 작업 폴더: /content
학습 CSV 크기: (866, 2)
테스트 CSV 크기: (217, 1)

실제 학습 이미지: 866
실제 테스트 이미지: 217

누락된 학습 이미지: 0
누락된 테스트 이미지: 0

이미지 크기: (128, 128)
이미지 모드: RGB

꽃 종류별 데이터 수:
label
petunia           206
passion_flower    201
wallflower        157
water_lily        155
watercress        147
Name: count, dtype: int64

결측치:
file_name    0
label        0
dtype: int64
file_name    0
dtype: int64

학습 CSV 예시:


,file_name,label
0,flower_00001.png,watercress
1,flower_00002.png,wallflower
2,flower_00003.png,petunia
3,flower_00004.png,passion_flower
4,flower_00005.png,passion_flower



테스트 CSV 예시:


,file_name
0,test_0001.png
1,test_0002.png
2,test_0003.png
3,test_0004.png
4,test_0005.png


In [15]:

import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split

SIZE = 64
CLASSES = sorted(train_csv_3['label'].unique())
class_to_idx = {name: i for i, name in enumerate(CLASSES)}

def load_images(df, folder):
    images = []

    for filename in df['file_name']:
        with Image.open(folder / filename) as img:
            img = img.convert('RGB')
            img = img.resize((SIZE, SIZE))
            arr = np.asarray(img, dtype=np.float32) / 255.0
            images.append(arr)

    return np.stack(images)

X_all_3 = load_images(train_csv_3, train_dir)
X_test_3 = load_images(test_csv_3, test_dir)

y_all_3 = train_csv_3['label'].map(class_to_idx).to_numpy()

X_train_3, X_valid_3, y_train_3, y_valid_3 = train_test_split(
    X_all_3,
    y_all_3,
    test_size=0.2,
    random_state=42,
    stratify=y_all_3
)

print('학습 이미지 배열:', X_train_3.shape)
print('검증 이미지 배열:', X_valid_3.shape)
print('테스트 이미지 배열:', X_test_3.shape)

print('\n픽셀 최솟값:', X_all_3.min())
print('픽셀 최댓값:', X_all_3.max())

print('\n데이터 타입:', X_all_3.dtype)
print('결측치:', np.isnan(X_all_3).sum())

print('\n클래스 번호 매핑:')
for name, idx in class_to_idx.items():
    print(idx, ':', name)

print('\n학습 클래스 분포:', np.bincount(y_train_3))
print('검증 클래스 분포:', np.bincount(y_valid_3))

print('\n테스트 이미지 수 일치:',
      len(X_test_3) == len(test_csv_3))


학습 이미지 배열: (692, 64, 64, 3)
검증 이미지 배열: (174, 64, 64, 3)
테스트 이미지 배열: (217, 64, 64, 3)

픽셀 최솟값: 0.0
픽셀 최댓값: 1.0

데이터 타입: float32
결측치: 0

클래스 번호 매핑:
0 : passion_flower
1 : petunia
2 : wallflower
3 : water_lily
4 : watercress

학습 클래스 분포: [161 165 125 124 117]
검증 클래스 분포: [40 41 32 31 30]

테스트 이미지 수 일치: True


In [16]:

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input, Conv2D, MaxPooling2D,
    Flatten, Dense, Dropout, BatchNormalization
)
from tensorflow.keras.optimizers import Adam

tf.keras.utils.set_random_seed(42)

model_3 = Sequential([
    Input(shape=(64, 64, 3)),

    Conv2D(32, (3, 3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D((2, 2)),

    Conv2D(64, (3, 3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D((2, 2)),

    Conv2D(128, (3, 3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D((2, 2)),

    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.4),
    Dense(5, activation='softmax')
])

model_3.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model_3.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 64, 64, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 64, 64, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 32, 32, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 32, 32, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 32, 32, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 16, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 16, 16, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 16, 16, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 8, 8, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 8192)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     1,048,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 5)              │           645 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,143,493 (4.36 MB)

 Trainable params: 1,143,045 (4.36 MB)

 Non-trainable params: 448 (1.75 KB)

In [17]:

import tensorflow as tf
import numpy as np
from pathlib import Path
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import (
    EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
)
from tensorflow.keras.models import load_model

checkpoint_path_3 = Path('candidate_3_best.keras')

if checkpoint_path_3.exists():
    raise FileExistsError('기존 최적 모델이 있습니다. 덮어쓰지 않습니다.')

augment_3 = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.15,
    horizontal_flip=True
)

train_generator_3 = augment_3.flow(
    X_train_3,
    y_train_3,
    batch_size=32,
    shuffle=True,
    seed=42
)

callbacks_3 = [
    ModelCheckpoint(
        str(checkpoint_path_3),
        monitor='val_accuracy',
        mode='max',
        save_best_only=True,
        verbose=1
    ),
    EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=4,
        min_lr=0.00001,
        verbose=1
    )
]

history_3 = model_3.fit(
    train_generator_3,
    validation_data=(X_valid_3, y_valid_3),
    epochs=50,
    callbacks=callbacks_3,
    verbose=1
)

best_model_3 = load_model(str(checkpoint_path_3))

valid_loss_3, valid_acc_3 = best_model_3.evaluate(
    X_valid_3,
    y_valid_3,
    verbose=0
)

print('\n실제 학습 Epoch:', len(history_3.history['loss']))
print('최고 기록 검증 정확도:',
      round(max(history_3.history['val_accuracy']), 4))
print('저장 모델 검증 정확도:', round(valid_acc_3, 4))
print('저장 모델 검증 손실:', round(valid_loss_3, 4))
print('최적 모델 파일 존재:', checkpoint_path_3.exists())


Epoch 1/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 646ms/step - accuracy: 0.4420 - loss: 2.8040
Epoch 1: val_accuracy improved from None to 0.32184, saving model to candidate_3_best.keras

Epoch 1: finished saving model to candidate_3_best.keras
22/22 ━━━━━━━━━━━━━━━━━━━━ 22s 698ms/step - accuracy: 0.5361 - loss: 2.0183 - val_accuracy: 0.3218 - val_loss: 3.6072 - learning_rate: 0.0010
Epoch 2/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 663ms/step - accuracy: 0.5991 - loss: 1.1228
Epoch 2: val_accuracy did not improve from 0.32184
22/22 ━━━━━━━━━━━━━━━━━━━━ 16s 726ms/step - accuracy: 0.6445 - loss: 0.9556 - val_accuracy: 0.2126 - val_loss: 7.5049 - learning_rate: 0.0010
Epoch 3/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 580ms/step - accuracy: 0.6978 - loss: 0.7823
Epoch 3: val_accuracy did not improve from 0.32184
22/22 ━━━━━━━━━━━━━━━━━━━━ 18s 611ms/step - accuracy: 0.6965 - loss: 0.8094 - val_accuracy: 0.1839 - val_loss: 10.0840 - learning_rate: 0.0010
Epoch 4/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 576ms/step - accuracy:

In [18]:

import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix
)

train_pred_3 = best_model_3.predict(
    X_train_3, verbose=0
).argmax(axis=1)

valid_pred_3 = best_model_3.predict(
    X_valid_3, verbose=0
).argmax(axis=1)

print('저장 모델 학습 정확도:',
      round(accuracy_score(y_train_3, train_pred_3), 4))

print('저장 모델 검증 정확도:',
      round(accuracy_score(y_valid_3, valid_pred_3), 4))

print('\n실제 검증 클래스 분포:')
print(pd.Series(y_valid_3).map(
    dict(enumerate(CLASSES))
).value_counts())

print('\n모델 예측 클래스 분포:')
print(pd.Series(valid_pred_3).map(
    dict(enumerate(CLASSES))
).value_counts())

print('\n클래스별 성능:')
print(classification_report(
    y_valid_3,
    valid_pred_3,
    target_names=CLASSES,
    zero_division=0,
    digits=4
))

print('\n혼동 행렬:')
display(pd.DataFrame(
    confusion_matrix(y_valid_3, valid_pred_3),
    index=CLASSES,
    columns=CLASSES
))


저장 모델 학습 정확도: 0.3945
저장 모델 검증 정확도: 0.3908

실제 검증 클래스 분포:
petunia           41
passion_flower    40
wallflower        32
water_lily        31
watercress        30
Name: count, dtype: int64

모델 예측 클래스 분포:
water_lily    85
wallflower    73
watercress    16
Name: count, dtype: int64

클래스별 성능:
                precision    recall  f1-score   support

passion_flower     0.0000    0.0000    0.0000        40
       petunia     0.0000    0.0000    0.0000        41
    wallflower     0.4384    1.0000    0.6095        32
    water_lily     0.2824    0.7742    0.4138        31
    watercress     0.7500    0.4000    0.5217        30

      accuracy                         0.3908       174
     macro avg     0.2941    0.4348    0.3090       174
  weighted avg     0.2602    0.3908    0.2758       174


혼동 행렬:


,passion_flower,petunia,wallflower,water_lily,watercress
passion_flower,0,0,8,29,3
petunia,0,0,8,32,1
wallflower,0,0,32,0,0
water_lily,0,0,7,24,0
watercress,0,0,18,0,12


In [19]:

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input, Conv2D, MaxPooling2D,
    GlobalAveragePooling2D, Dense, Dropout
)
from tensorflow.keras.optimizers import Adam

tf.keras.utils.set_random_seed(42)

model_3_v2 = Sequential([
    Input(shape=(64, 64, 3)),

    Conv2D(32, 3, activation='relu', padding='same'),
    MaxPooling2D(2),

    Conv2D(64, 3, activation='relu', padding='same'),
    MaxPooling2D(2),

    Conv2D(128, 3, activation='relu', padding='same'),
    MaxPooling2D(2),

    Conv2D(128, 3, activation='relu', padding='same'),

    GlobalAveragePooling2D(),

    Dense(64, activation='relu'),
    Dropout(0.3),

    Dense(5, activation='softmax')
])

model_3_v2.compile(
    optimizer=Adam(learning_rate=0.0003),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model_3_v2.summary()

print('\n기존 모델 파라미터:', model_3.count_params())
print('개선 모델 파라미터:', model_3_v2.count_params())
print('파라미터 감소율:',
      round(
          (1 - model_3_v2.count_params() /
           model_3.count_params()) * 100, 2
      ), '%')


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_3 (Conv2D)               │ (None, 64, 64, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 32, 32, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 32, 32, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 16, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 16, 16, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 8, 8, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ (None, 8, 8, 128)      │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 5)              │           325 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 249,413 (974.27 KB)

 Trainable params: 249,413 (974.27 KB)

 Non-trainable params: 0 (0.00 B)


기존 모델 파라미터: 1143493
개선 모델 파라미터: 249413
파라미터 감소율: 78.19 %


In [20]:

import numpy as np
from pathlib import Path
from tensorflow.keras.callbacks import (
    EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
)
from tensorflow.keras.models import load_model

checkpoint_v2 = Path('candidate_3_v2_best.keras')

if checkpoint_v2.exists():
    raise FileExistsError('기존 개선 모델 파일을 보호합니다.')

callbacks_v2 = [
    ModelCheckpoint(
        str(checkpoint_v2),
        monitor='val_accuracy',
        mode='max',
        save_best_only=True,
        verbose=1
    ),
    EarlyStopping(
        monitor='val_loss',
        patience=12,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=0.00001,
        verbose=1
    )
]

train_generator_v2 = augment_3.flow(
    X_train_3,
    y_train_3,
    batch_size=32,
    shuffle=True,
    seed=42
)

history_v2 = model_3_v2.fit(
    train_generator_v2,
    validation_data=(X_valid_3, y_valid_3),
    epochs=60,
    callbacks=callbacks_v2,
    verbose=1
)

best_model_v2 = load_model(str(checkpoint_v2))

train_loss_v2, train_acc_v2 = best_model_v2.evaluate(
    X_train_3,
    y_train_3,
    verbose=0
)

valid_loss_v2, valid_acc_v2 = best_model_v2.evaluate(
    X_valid_3,
    y_valid_3,
    verbose=0
)

print('\n실제 학습 Epoch:', len(history_v2.history['loss']))
print('최고 기록 검증 정확도:',
      round(max(history_v2.history['val_accuracy']), 4))
print('저장 모델 학습 정확도:', round(train_acc_v2, 4))
print('저장 모델 검증 정확도:', round(valid_acc_v2, 4))
print('저장 모델 검증 손실:', round(valid_loss_v2, 4))
print('모델 파일 존재:', checkpoint_v2.exists())


Epoch 1/60
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 553ms/step - accuracy: 0.2421 - loss: 1.5806
Epoch 1: val_accuracy improved from None to 0.36782, saving model to candidate_3_v2_best.keras

Epoch 1: finished saving model to candidate_3_v2_best.keras
22/22 ━━━━━━━━━━━━━━━━━━━━ 17s 601ms/step - accuracy: 0.2746 - loss: 1.5440 - val_accuracy: 0.3678 - val_loss: 1.3927 - learning_rate: 3.0000e-04
Epoch 2/60
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 473ms/step - accuracy: 0.3719 - loss: 1.3154
Epoch 2: val_accuracy improved from 0.36782 to 0.37931, saving model to candidate_3_v2_best.keras

Epoch 2: finished saving model to candidate_3_v2_best.keras
22/22 ━━━━━━━━━━━━━━━━━━━━ 11s 503ms/step - accuracy: 0.3916 - loss: 1.2592 - val_accuracy: 0.3793 - val_loss: 1.1238 - learning_rate: 3.0000e-04
Epoch 3/60
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 471ms/step - accuracy: 0.3798 - loss: 1.2131
Epoch 3: val_accuracy improved from 0.37931 to 0.43103, saving model to candidate_3_v2_best.keras

Epoch 3: finished saving model to ca

In [21]:

import shutil
import numpy as np
import pandas as pd
from pathlib import Path
from tensorflow.keras.models import load_model
from sklearn.metrics import accuracy_score

source_3 = Path('candidate_3_v2_best.keras')
model_file_3 = Path('candidate_3.keras')
csv_file_3 = Path('candidate_3.csv')

if not source_3.exists():
    raise FileNotFoundError(source_3)

if model_file_3.exists() or csv_file_3.exists():
    raise FileExistsError('기존 후보 파일을 보호합니다.')

shutil.copy2(source_3, model_file_3)

loaded_3 = load_model(str(model_file_3), compile=False)

original_pred_3 = best_model_v2.predict(
    X_valid_3, verbose=0
)
loaded_pred_3 = loaded_3.predict(
    X_valid_3, verbose=0
)

print('저장 전후 예측 확률 일치:',
      np.allclose(original_pred_3, loaded_pred_3))

valid_labels_3 = loaded_pred_3.argmax(axis=1)

print('재불러온 모델 검증 정확도:',
      round(accuracy_score(y_valid_3, valid_labels_3), 4))

test_probs_3 = loaded_3.predict(X_test_3, verbose=0)

assert test_probs_3.shape == (len(test_csv_3), 5)
assert np.isfinite(test_probs_3).all()

test_labels_3 = np.array(CLASSES)[
    test_probs_3.argmax(axis=1)
]

submission_3 = test_csv_3.copy()
submission_3['label'] = test_labels_3

submission_3.to_csv(
    csv_file_3,
    index=False,
    encoding='utf-8-sig'
)

check_3 = pd.read_csv(csv_file_3)

print('\n모델 파일 생성:', model_file_3.exists())
print('CSV 파일 생성:', csv_file_3.exists())
print('CSV 크기:', check_3.shape)

print('컬럼 구조 일치:',
      check_3.columns.tolist() == ['file_name', 'label'])

print('테스트 행 순서 유지:',
      check_3['file_name'].equals(test_csv_3['file_name']))

print('예측 결측치:', check_3['label'].isna().sum())

print('예측 라벨 정상:',
      check_3['label'].isin(CLASSES).all())

print('모델과 CSV 예측 일치:',
      np.array_equal(check_3['label'].to_numpy(), test_labels_3))

print('\n꽃 종류별 예측 분포:')
print(check_3['label'].value_counts())

display(check_3.head())


저장 전후 예측 확률 일치: True
재불러온 모델 검증 정확도: 0.8621

모델 파일 생성: True
CSV 파일 생성: True
CSV 크기: (217, 2)
컬럼 구조 일치: True
테스트 행 순서 유지: True
예측 결측치: 0
예측 라벨 정상: True
모델과 CSV 예측 일치: True

꽃 종류별 예측 분포:
label
petunia           55
passion_flower    52
wallflower        46
water_lily        35
watercress        29
Name: count, dtype: int64


,file_name,label
0,test_0001.png,passion_flower
1,test_0002.png,wallflower
2,test_0003.png,passion_flower
3,test_0004.png,petunia
4,test_0005.png,water_lily


In [22]:

from pathlib import Path
import shutil
import numpy as np
import pandas as pd
from tensorflow.keras.models import load_model
from sklearn.metrics import accuracy_score

files_3 = {
    'candidate_3.keras': 'answer_3.keras',
    'candidate_3.csv': 'answer_3.csv'
}

for src, dst in files_3.items():
    if not Path(src).exists():
        raise FileNotFoundError(src)
    if Path(dst).exists():
        raise FileExistsError(f'기존 파일 보호: {dst}')

for src, dst in files_3.items():
    shutil.copy2(src, dst)
    print('제출 파일 생성:', dst)

saved_model_3 = load_model('answer_3.keras', compile=False)
answer_3 = pd.read_csv('answer_3.csv')

valid_pred = saved_model_3.predict(
    X_valid_3, verbose=0
).argmax(axis=1)

test_pred = saved_model_3.predict(
    X_test_3, verbose=0
).argmax(axis=1)

test_labels = np.array(CLASSES)[test_pred]

print('\n모델 검증 정확도:',
      round(accuracy_score(y_valid_3, valid_pred), 4))

print('CSV 크기:', answer_3.shape)

print('컬럼 구조 일치:',
      answer_3.columns.tolist() == ['file_name', 'label'])

print('테스트 행 순서 일치:',
      answer_3['file_name'].equals(test_csv_3['file_name']))

print('모델과 CSV 예측 일치:',
      np.array_equal(answer_3['label'].to_numpy(), test_labels))

print('예측 결측치:', answer_3['label'].isna().sum())

print('예측 라벨 정상:',
      answer_3['label'].isin(CLASSES).all())

print('\n전체 제출 파일 확인:')
for filename in [
    'answer_1.pkl', 'answer_1.csv',
    'answer_2.pkl', 'answer_2.csv',
    'answer_3.keras', 'answer_3.csv'
]:
    print(filename, Path(filename).exists())


제출 파일 생성: answer_3.keras
제출 파일 생성: answer_3.csv

모델 검증 정확도: 0.8621
CSV 크기: (217, 2)
컬럼 구조 일치: True
테스트 행 순서 일치: True
모델과 CSV 예측 일치: True
예측 결측치: 0
예측 라벨 정상: True

전체 제출 파일 확인:
answer_1.pkl True
answer_1.csv True
answer_2.pkl True
answer_2.csv True
answer_3.keras True
answer_3.csv True


<details><summary>힌트 1단계 (접근 방향) - 3번 문항</summary>

압축을 풀고 폴더 구조(학습: 클래스별 폴더, 테스트: 한 폴더)를 확인합니다. 이미지를 같은 크기로 읽어 0~1 로 맞추고, 검증 데이터를 떼어 **작은 CNN**(또는 사전학습 모델 특징 + 분류기)을 학습합니다. 증강·Dropout·조기 종료로 과적합을 막고, 테스트는 `03_test_x.csv` 순서대로 예측하세요.

- 이 문항의 핵심: 학습 이미지가 클래스별 폴더가 아니라 한 폴더에 있고 라벨은 CSV 에 있습니다. CSV 의 `file_name` 으로 `load_img` 해서 배열로 만드세요.

</details>

<details><summary>힌트 2단계 (사용할 도구) - 3번 문항</summary>

사용할 도구: `tensorflow`, `pandas`, `train_test_split`, `load_img`, `img_to_array`, `Sequential`, `load_model`, `Input`, `Conv2D`, `MaxPooling2D`, `Flatten`, `Dense`, `Dropout`, `EarlyStopping`.

- 참고 문서: <https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html>
- 참고 문서: <https://keras.io/api/data_loading/image/#loadimg-function>
- 참고 문서: <https://keras.io/api/data_loading/image/#imgtoarray-function>
- 참고 문서: <https://keras.io/api/models/sequential/>
- 참고 문서: <https://keras.io/api/layers/convolution_layers/convolution2d/>
- 참고 문서: <https://keras.io/api/layers/pooling_layers/max_pooling2d/>

</details>

<details><summary>힌트 3단계 (코드 구조) - 3번 문항</summary>

코드의 **앞부분은 구조만** 보여 줍니다(`____` 자리는 가이드와 공식 문서를 보고 채우세요). **뒷부분은 주석으로 할 일만** 적었습니다 — 주석을 따라 직접 작성하세요.
```python
import ____
import ____
IMG_DIR = os.environ.get('____', '____')        # 압축을 풀 위치(브라우저판은 메모리 폴더 /tmp)
if not os.path.exists(os.path.join(IMG_DIR, '____')):
    zipfile.ZipFile('03_images.zip').extractall(IMG_DIR)
import ____
from sklearn.model_selection import ____
from tensorflow.keras.utils import ____, ____
from tensorflow.keras.models import ____, ____
from tensorflow.keras.layers import ____, ____, ____, ____, ____, ____
from tensorflow.keras.callbacks import ____, ____, ____

train = pd.read_csv('03_train.csv')
test = pd.read_csv('03_test_x.csv')
SIZE = ____                                                 # 브라우저 메모리·속도를 위해 ____×____ 로 읽음
CLASSES = sorted(train['____'].unique())                 # 라벨 번호 = 알파벳 순
# 학습 이미지가 한 폴더(03_train/)에 있고 라벨은 CSV 에만 있음 → CSV 의 파일명으로 한 장씩 읽음

# ▷ 모델 구성(CNN 또는 사전학습 특징 추출)
# ▷ 학습(검증 데이터·조기 종료)
# ▷ 모델 파일 저장 — answer_N.pkl / .h5
# ▷ 테스트 데이터 예측(행 순서 그대로)
# ▷ 예측 CSV 저장 — answer_N.csv(테스트 열 + 마지막 정답 열)
```
- 제출 파일명: `answer_3.h5`(모델) · `answer_3.csv`(예측)

</details>